# 04 · Statistical analysis
Formal tests use one value per city (independent units); places sharing an identical model series are reduced to one representative.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 30, "display.width", 160)

from src.data import load
from src.analysis import descriptive, trends, seasonality, correlations, statistical_tests as st, data_quality as dq, sensitivity
from src.visualization import charts, maps
from src.data.transform import independent_units

wb = descriptive.real_only(load.load_processed("wb_daily"))
cities = load.load_processed("wb_cities")

In [2]:
iu = independent_units(wb, cities)
print(wb['city_id'].nunique(), '->', iu['city_id'].nunique(), 'distinct series')

339 -> 324 distinct series


## Seasonal differences (Friedman, cities as blocks)

In [3]:
r = st.seasonal_friedman(iu)
print(r.interpretation)
r.details.round(4)

Across 324 cities, p < 0.001 (below α); Kendall's W = 0.64 (large agreement in seasonal ranking). Highest median season: Post-monsoon (75.5 µg/m³); lowest: Monsoon (50.7 µg/m³). 10 of 10 pairwise season contrasts remain below α after Holm correction.


,season_a,season_b,median_diff,rank_biserial,W,p_value,p_holm
0,Winter,Spring,13.7673,0.9935,171.0,0.0,0.0
1,Winter,Pre-monsoon,7.3978,0.7476,6644.0,0.0,0.0
2,Winter,Monsoon,19.6062,0.9560,1158.0,0.0,0.0
3,Winter,Post-monsoon,-2.3145,-0.2795,18966.0,0.0,0.0
4,Spring,Pre-monsoon,-4.3995,-0.4156,15384.0,0.0,0.0
5,Spring,Monsoon,5.5506,0.7157,7483.0,0.0,0.0
6,Spring,Post-monsoon,-17.8740,-0.9979,55.0,0.0,0.0
7,Pre-monsoon,Monsoon,10.1358,0.9869,346.0,0.0,0.0
8,Pre-monsoon,Post-monsoon,-12.9370,-0.9424,1517.0,0.0,0.0
9,Monsoon,Post-monsoon,-25.9194,-0.9813,492.0,0.0,0.0


## Province differences: Kruskal–Wallis and assumption-gated ANOVA

In [4]:
cm = iu.groupby(['city_id','province'], observed=True)['pm25'].median().reset_index()
k = st.group_kruskal(cm, 'pm25'); print(k.interpretation)
a = st.group_anova_if_justified(iu.groupby(['city_id','province'], observed=True)['pm25'].mean().reset_index(), 'pm25')
print(a.interpretation); a.assumption_checks

4 groups, 322 cities: H = 111.10, p < 0.001 (below α); ε² = 0.34 (large). 5 of 6 pairwise contrasts below α after Holm.


ANOVA not reported because its assumptions are not reasonably met: normality rejected (Shapiro–Wilk p < 0.05) in: Balochistan, Punjab, Sindh; equal variances rejected (Levene p = 0.000626). Use the Kruskal–Wallis result instead.


{'shapiro_p_by_group': {'Balochistan': 0.012764750889014508,
  'Khyber Pakhtunkhwa': 0.06353674732219917,
  'Punjab': 0.04276647344021045,
  'Sindh': 0.019776178913298356},
 'levene_p': 0.0006263863898661343}

## Weekday vs weekend — significance vs importance

In [5]:
w = st.weekday_weekend_test(iu)
print(w.interpretation)

Median weekday−weekend difference = -0.23 µg/m³ (95 % bootstrap CI -0.30 to -0.16), i.e. -0.3 % of the typical weekend level; p < 0.001 (below α); rank-biserial = -0.40 (moderate consistency across cities). The difference is consistent but very small in absolute terms; statistical significance here does not imply practical importance.


## Trends
Deseasonalised monthly anomalies, OLS with Newey–West CIs, Theil–Sen, BH-FDR.

In [6]:
tr = trends.city_trends(wb).merge(cities[['city_id','artifact_flag']], on='city_id')
pd.crosstab(tr['direction'], tr['artifact_flag'], margins=True)

artifact_flag,False,True,All
direction,,,
decreasing,30,105,135
increasing,60,1,61
no clear trend,137,6,143
All,227,112,339


In [7]:
charts.trend_forest(tr, n=10)

In [8]:
yrs = trends.complete_years(iu)
for sub, label in ((iu, 'all'), (iu[~iu['city_id'].astype(str).isin(set(cities.loc[cities['artifact_flag'],'city_id'].astype(str)))], 'artifact-flagged excluded')):
    t = st.period_change_test(trends.annual_series(sub), yrs[0], yrs[-1])
    print(label, '->', t.interpretation)

all -> Median city change -10.9 % (95 % bootstrap CI -17.1 to -5.6 %); 217 cities decreased, 107 increased; p < 0.001 (below α).


artifact-flagged excluded -> Median city change -0.7 % (95 % bootstrap CI -2.4 to +0.8 %); 116 cities decreased, 101 increased; p = 0.037 (below α).


## Weather correlations (raw vs deseasonalised, autocorrelation-adjusted)

In [9]:
ww = load.load_processed('wb_weather_daily')
rows = []
for x in ['temperature','humidity','wind_speed','precipitation']:
    for des in (False, True):
        rows.append(correlations.pooled_city_correlation(ww, x, deseasonalize=des))
pd.DataFrame(rows)

,x,y,method,deseasonalized,n_cities,median_r,min_r,max_r,n_obs_total,n_significant_fdr,n_positive,n_negative
0,temperature,pm25,spearman,False,22,-0.164165,-0.519609,-0.019975,61534,14,0,22
1,temperature,pm25,spearman,True,22,0.206030,-0.053366,0.306944,61534,18,20,2
2,humidity,pm25,spearman,False,22,-0.196774,-0.520015,0.080475,61534,19,2,20
3,humidity,pm25,spearman,True,22,-0.281304,-0.426872,-0.090267,61534,22,0,22
4,wind_speed,pm25,spearman,False,22,-0.326090,-0.458328,0.446533,61534,21,1,21
5,wind_speed,pm25,spearman,True,22,-0.251186,-0.346549,0.175245,61534,21,1,21
6,precipitation,pm25,spearman,False,22,-0.460903,-0.527109,-0.319912,61534,22,0,22
7,precipitation,pm25,spearman,True,22,-0.258677,-0.326978,-0.134892,61534,22,0,22


In [10]:
charts.lag_chart(correlations.lagged_correlation(ww, 'wind_speed', deseasonalize=True), 'wind_speed')

In [11]:
corr, n = correlations.correlation_matrix(correlations.add_anomalies(ww, ['pm25','temperature','humidity','wind_speed','precipitation']),
    ['pm25_anom','temperature_anom','humidity_anom','wind_speed_anom','precipitation_anom'])
charts.correlation_heatmap(corr, n)

## Sensitivity analysis

In [12]:
sensitivity.scenario_metrics(wb, cities).round(2)

,scenario,rows,rows_removed_pct,cities,overall_median,overall_mean,median_of_city_medians,top_city,top10_overlap_with_baseline,rank_spearman_vs_baseline
0,Baseline (all retained data),948183,0.00,339,59.38,68.36,62.68,Larkana,10,1.00
1,Exclude flagged outliers,947400,0.08,339,59.37,68.36,62.68,Larkana,10,1.00
2,Winsorise at 1st/99th percentile,948183,0.00,339,59.38,68.00,62.68,Larkana,10,1.00
3,Complete calendar years only,838347,11.58,339,60.50,69.53,63.94,Larkana,10,1.00
4,Last 3 complete years only,359340,62.10,339,57.83,63.87,60.86,Rahim Yar Khan,5,0.95
5,Drop winter months (Dec–Feb),711561,24.96,339,57.09,64.84,59.28,Larkana,9,0.99
6,One place per identical model series,906228,4.42,324,58.78,67.80,61.73,Larkana,8,1.00
7,Exclude model-artifact-flagged places,634919,33.04,227,53.81,59.13,57.68,Khurarianwala,0,1.00
